<a href="https://colab.research.google.com/github/Rini43/Case_Study/blob/main/Case_Study_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 12.1 MB/s eta 0:00:00


# Libraries

In [76]:
import pandas as pd
import numpy as np
import string
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import Word2Vec
import gensim.downloader as api

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score,
    classification_report,confusion_matrix)

from sklearn.model_selection import GridSearchCV

from tensorflow.keras.layers import (Input, Dense, Dropout,
    BatchNormalization, Concatenate, LeakyReLU)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Read the Data

In [4]:
filepath = "/content/drive/MyDrive/Data/donor.csv"
df_donor = pd.read_csv(filepath)
df_donor.head(5)

,id,teacher_prefix,school_state,project_grade_category,project_subject_categories,project_subject_subcategories,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary
0,p253737,mrs,in,grades_prek_2,literacy_language,esl_literacy,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students_need_opportunities_practice_beginning...,0
1,p258326,mr,fl,grades_6_8,history_civics_health_sports,civics_government_teamsports,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students_need_projector_help_viewing_education...,0
2,p182444,ms,az,grades_6_8,health_sports,health_wellness_teamsports,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students_need_shine_guards_athletic_socks_socc...,0
3,p246581,mrs,ky,grades_prek_2,literacy_language_math_science,literacy_mathematics,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students_need_engage_reading_math_way_inspire_...,0
4,p104768,mrs,tx,grades_prek_2,math_science,mathematics,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students_need_hands_practice_mathematics_fun_p...,0


# Exploratory Data Analysis

In [5]:
df_donor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109248 entries, 0 to 109247
Data columns (total 14 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   id                                            109248 non-null  object 
 1   teacher_prefix                                109248 non-null  object 
 2   school_state                                  109248 non-null  object 
 3   project_grade_category                        109248 non-null  object 
 4   project_subject_categories                    109248 non-null  object 
 5   project_subject_subcategories                 109248 non-null  object 
 6   teacher_number_of_previously_posted_projects  109248 non-null  int64  
 7   project_is_approved                           109248 non-null  int64  
 8   price                                         109248 non-null  float64
 9   quantity                                      10

In [6]:
df_donor.describe()

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,isdigit_summary
count,109248.000000,109248.000000,109248.000000,109248.000000,109248.000000
mean,11.153165,0.848583,298.119343,16.965610,0.144222
std,27.777154,0.358456,367.498030,26.182942,0.351317
min,0.000000,0.000000,0.660000,1.000000,0.000000
25%,0.000000,1.000000,104.310000,4.000000,0.000000
50%,2.000000,1.000000,206.220000,9.000000,0.000000
75%,9.000000,1.000000,379.000000,21.000000,0.000000
max,451.000000,1.000000,9999.000000,930.000000,1.000000


In [7]:
df_donor.isnull().sum()

,0
id,0
teacher_prefix,0
school_state,0
project_grade_category,0
project_subject_categories,0
project_subject_subcategories,0
teacher_number_of_previously_posted_projects,0
project_is_approved,0
price,0
quantity,0


In [8]:
df_donor.duplicated().sum()

np.int64(0)

In [9]:
df_donor = df_donor.drop(columns=['id'])

In [10]:
# Handling the missing value

df_donor['cleaned_titles'] = df_donor['cleaned_titles'].fillna('')

In [11]:
# After handling the missing value

df_donor['cleaned_titles'].isnull().sum()

np.int64(0)

In [12]:
df_donor["project_is_approved"].value_counts()

,count
project_is_approved,
1,92706
0,16542


In [13]:
df_donor['project_subject_categories'].nunique()

51

In [14]:
df_donor['project_grade_category'].unique()

array(['grades_prek_2', 'grades_6_8', 'grades_3_5', 'grades_9_12'],
      dtype=object)

In [15]:
df_donor['project_subject_subcategories'].nunique()

401

We are taking these as Categorical encoding

* teacher_prefix
* school_state
* project_grade_category
* project_subject_categories
* project_subject_subcategories

Keep numerical features as they are

* price
* quantity
* teacher_number_of_previously_posted_projects

# Encoding

In [16]:
categorical_columns = [
    "teacher_prefix",
    "school_state",
    "project_grade_category",
    "project_subject_categories",
    "project_subject_subcategories"
]

df_donor = pd.get_dummies(
    df_donor,
    columns=categorical_columns,
    drop_first=True
)

df_donor.head()

# No artificial ordering between categories
# Works well with most machine learning algorithms.

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_socialsciences_visualarts,project_subject_subcategories_specialneeds,project_subject_subcategories_specialneeds_teamsports,project_subject_subcategories_specialneeds_visualarts,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students_need_opportunities_practice_beginning...,0,False,True,...,False,False,False,False,False,False,False,False,False,False
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students_need_projector_help_viewing_education...,0,True,False,...,False,False,False,False,False,False,False,False,False,False
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students_need_shine_guards_athletic_socks_socc...,0,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students_need_engage_reading_math_way_inspire_...,0,False,True,...,False,False,False,False,False,False,False,False,False,False
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students_need_hands_practice_mathematics_fun_p...,0,False,True,...,False,False,False,False,False,False,False,False,False,False


* Label Encoding do if the model requires integer-encoded categories or if there are many unique categories and you want to save memory.

# NLP Preprocessing

In [17]:
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

## Punctuations

In [18]:
# Select all object (string) columns

text_columns = df_donor.select_dtypes(include='object').columns

for col in text_columns:
    if col == 'cleaned_summary':
        continue

    df_donor[col] = df_donor[col].str.translate(
        str.maketrans('', '', string.punctuation)
    )

df_donor.head()

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_socialsciences_visualarts,project_subject_subcategories_specialneeds,project_subject_subcategories_specialneeds_teamsports,project_subject_subcategories_specialneeds_visualarts,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students_need_opportunities_practice_beginning...,0,False,True,...,False,False,False,False,False,False,False,False,False,False
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students_need_projector_help_viewing_education...,0,True,False,...,False,False,False,False,False,False,False,False,False,False
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students_need_shine_guards_athletic_socks_socc...,0,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students_need_engage_reading_math_way_inspire_...,0,False,True,...,False,False,False,False,False,False,False,False,False,False
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students_need_hands_practice_mathematics_fun_p...,0,False,True,...,False,False,False,False,False,False,False,False,False,False


# HTML tags

In [19]:
# Checking for HTML tags

# Find object (text) columns
text_columns = df_donor.select_dtypes(include='object').columns

for col in text_columns:
    html_count = df_donor[col].astype(str).str.contains(r'<[^>]+>', regex=True, na=False).sum()
    print(f"{col}: {html_count} rows contain HTML tags")

cleaned_titles: 0 rows contain HTML tags
cleaned_essays: 0 rows contain HTML tags
cleaned_summary: 0 rows contain HTML tags


## Lowercase

In [20]:
# Checking if there is any Upper Case Values

text_columns = df_donor.select_dtypes(include='object').columns

for col in text_columns:
    has_uppercase = df_donor[col].astype(str).str.contains(r'[A-Z]', regex=True).any()
    print(f"{col}: {has_uppercase}")

# There is no Uppercase values so no need for this step

cleaned_titles: False
cleaned_essays: False
cleaned_summary: False


## Tokenziation

In [21]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [22]:
# Tokenize all text columns

text_columns = ['cleaned_titles', 'cleaned_essays', 'cleaned_summary']

for col in text_columns:

    # Replace underscores with spaces
    df_donor[col] = df_donor[col].str.replace('_', ' ', regex=False)

    # Tokenize
    df_donor[col.replace('cleaned_', '') + '_tokens'] = df_donor[col].apply(nltk.word_tokenize)

df_donor.head()

# These are fixed categories. There are only a few possible values. So there is no need to do tokenization for "project_grade_category"
# These are labels, not sentences. So there is no need to do tokenization for "project_subject_categories"
# These are labels, not sentences. So there is no need to do tokenization for "project_subject_subcategories"

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_specialneeds_visualarts,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger,titles_tokens,essays_tokens,summary_tokens
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students need opportunities practice beginning...,0,False,True,...,False,False,False,False,False,False,False,"[educational, support, english, learners, home]","[students, english, learners, working, english...","[students, need, opportunities, practice, begi..."
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students need projector help viewing education...,0,True,False,...,False,False,False,False,False,False,False,"[wanted, projector, hungry, learners]","[students, arrive, school, eager, learn, polit...","[students, need, projector, help, viewing, edu..."
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students need shine guards athletic socks socc...,0,False,False,...,False,False,False,False,False,False,False,"[soccer, equipment, awesome, middle, school, s...","[true, champions, not, always, ones, win, guts...","[students, need, shine, guards, athletic, sock..."
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students need engage reading math way inspire ...,0,False,True,...,False,False,False,False,False,False,False,"[techie, kindergarteners]","[work, unique, school, filled, esl, english, s...","[students, need, engage, reading, math, way, i..."
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students need hands practice mathematics fun p...,0,False,True,...,False,False,False,False,False,False,False,"[interactive, math, tools]","[second, grade, classroom, next, year, made, a...","[students, need, hands, practice, mathematics,..."


## Stop Words Removal

In [23]:
stop_words = set(stopwords.words("english"))

for col in ["titles_tokens", "essays_tokens", "summary_tokens"]:
    df_donor[col] = df_donor[col].apply(
        lambda tokens: [word for word in tokens if word.lower() not in stop_words])

df_donor.head()

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_specialneeds_visualarts,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger,titles_tokens,essays_tokens,summary_tokens
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students need opportunities practice beginning...,0,False,True,...,False,False,False,False,False,False,False,"[educational, support, english, learners, home]","[students, english, learners, working, english...","[students, need, opportunities, practice, begi..."
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students need projector help viewing education...,0,True,False,...,False,False,False,False,False,False,False,"[wanted, projector, hungry, learners]","[students, arrive, school, eager, learn, polit...","[students, need, projector, help, viewing, edu..."
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students need shine guards athletic socks socc...,0,False,False,...,False,False,False,False,False,False,False,"[soccer, equipment, awesome, middle, school, s...","[true, champions, always, ones, win, guts, mia...","[students, need, shine, guards, athletic, sock..."
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students need engage reading math way inspire ...,0,False,True,...,False,False,False,False,False,False,False,"[techie, kindergarteners]","[work, unique, school, filled, esl, english, s...","[students, need, engage, reading, math, way, i..."
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students need hands practice mathematics fun p...,0,False,True,...,False,False,False,False,False,False,False,"[interactive, math, tools]","[second, grade, classroom, next, year, made, a...","[students, need, hands, practice, mathematics,..."


* categorical variables, but they are actually text-like categorical features because they contain words or phrases
* These values are labels, not sentences.
* They contain very few words.
* They rarely include common stop words like "the", "is", "and", or "to".

## Stemming

In [24]:
stemmer = PorterStemmer()

# Apply stemming

for col in ["titles_tokens", "essays_tokens", "summary_tokens"]:
    df_donor[col] = df_donor[col].apply(
        lambda tokens: [stemmer.stem(word) for word in tokens]
    )

df_donor.head()

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_specialneeds_visualarts,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger,titles_tokens,essays_tokens,summary_tokens
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students need opportunities practice beginning...,0,False,True,...,False,False,False,False,False,False,False,"[educ, support, english, learner, home]","[student, english, learner, work, english, sec...","[student, need, opportun, practic, begin, read..."
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students need projector help viewing education...,0,True,False,...,False,False,False,False,False,False,False,"[want, projector, hungri, learner]","[student, arriv, school, eager, learn, polit, ...","[student, need, projector, help, view, educ, p..."
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students need shine guards athletic socks socc...,0,False,False,...,False,False,False,False,False,False,False,"[soccer, equip, awesom, middl, school, student]","[true, champion, alway, one, win, gut, mia, ha...","[student, need, shine, guard, athlet, sock, so..."
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students need engage reading math way inspire ...,0,False,True,...,False,False,False,False,False,False,False,"[techi, kindergarten]","[work, uniqu, school, fill, esl, english, seco...","[student, need, engag, read, math, way, inspir..."
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students need hands practice mathematics fun p...,0,False,True,...,False,False,False,False,False,False,False,"[interact, math, tool]","[second, grade, classroom, next, year, made, a...","[student, need, hand, practic, mathemat, fun, ..."


## Lemmatization

In [25]:
lemmatizer = WordNetLemmatizer()

# Apply lemmatization

for col in ["titles_tokens", "essays_tokens", "summary_tokens"]:
    df_donor[col] = df_donor[col].apply(
        lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
    )

df_donor.head()

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_specialneeds_visualarts,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger,titles_tokens,essays_tokens,summary_tokens
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students need opportunities practice beginning...,0,False,True,...,False,False,False,False,False,False,False,"[educ, support, english, learner, home]","[student, english, learner, work, english, sec...","[student, need, opportun, practic, begin, read..."
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students need projector help viewing education...,0,True,False,...,False,False,False,False,False,False,False,"[want, projector, hungri, learner]","[student, arriv, school, eager, learn, polit, ...","[student, need, projector, help, view, educ, p..."
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students need shine guards athletic socks socc...,0,False,False,...,False,False,False,False,False,False,False,"[soccer, equip, awesom, middl, school, student]","[true, champion, alway, one, win, gut, mia, ha...","[student, need, shine, guard, athlet, sock, so..."
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students need engage reading math way inspire ...,0,False,True,...,False,False,False,False,False,False,False,"[techi, kindergarten]","[work, uniqu, school, fill, esl, english, seco...","[student, need, engag, read, math, way, inspir..."
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students need hands practice mathematics fun p...,0,False,True,...,False,False,False,False,False,False,False,"[interact, math, tool]","[second, grade, classroom, next, year, made, a...","[student, need, hand, practic, mathemat, fun, ..."


## Bag of Words

In [26]:
# Create combined text
df_donor["combined_text"] = (
    df_donor["titles_tokens"].apply(lambda x: " ".join(x)) + " " +
    df_donor["essays_tokens"].apply(lambda x: " ".join(x)) + " " +
    df_donor["summary_tokens"].apply(lambda x: " ".join(x))
)

# Create Bag of Words (BoW)
vectorizer = CountVectorizer(
    max_features=10000,
    min_df=5,
    max_df=0.9
)

# Convert text into sparse BoW matrix
X_bow = vectorizer.fit_transform(df_donor["combined_text"])

print("BoW Shape:", X_bow.shape)
print("Vocabulary Size:", len(vectorizer.vocabulary_))

BoW Shape: (109248, 10000)
Vocabulary Size: 10000


In [27]:
df_donor.head(5)

,teacher_number_of_previously_posted_projects,project_is_approved,price,quantity,cleaned_titles,cleaned_essays,cleaned_summary,isdigit_summary,teacher_prefix_mr,teacher_prefix_mrs,...,project_subject_subcategories_specialneeds_warmth_care_hunger,project_subject_subcategories_teamsports,project_subject_subcategories_teamsports_visualarts,project_subject_subcategories_visualarts,project_subject_subcategories_visualarts_warmth_care_hunger,project_subject_subcategories_warmth_care_hunger,titles_tokens,essays_tokens,summary_tokens,combined_text
0,0,0,154.60,23,educational support english learners home,students english learners working english seco...,students need opportunities practice beginning...,0,False,True,...,False,False,False,False,False,False,"[educ, support, english, learner, home]","[student, english, learner, work, english, sec...","[student, need, opportun, practic, begin, read...",educ support english learner home student engl...
1,7,1,299.00,1,wanted projector hungry learners,students arrive school eager learn polite gene...,students need projector help viewing education...,0,True,False,...,False,False,False,False,False,False,"[want, projector, hungri, learner]","[student, arriv, school, eager, learn, polit, ...","[student, need, projector, help, view, educ, p...",want projector hungri learner student arriv sc...
2,1,0,516.85,22,soccer equipment awesome middle school students,true champions not always ones win guts mia ha...,students need shine guards athletic socks socc...,0,False,False,...,False,False,False,False,False,False,"[soccer, equip, awesom, middl, school, student]","[true, champion, alway, one, win, gut, mia, ha...","[student, need, shine, guard, athlet, sock, so...",soccer equip awesom middl school student true ...
3,4,1,232.90,4,techie kindergarteners,work unique school filled esl english second l...,students need engage reading math way inspire ...,0,False,True,...,False,False,False,False,False,False,"[techi, kindergarten]","[work, uniqu, school, fill, esl, english, seco...","[student, need, engag, read, math, way, inspir...",techi kindergarten work uniqu school fill esl ...
4,1,1,67.98,4,interactive math tools,second grade classroom next year made around 2...,students need hands practice mathematics fun p...,0,False,True,...,False,False,False,False,False,False,"[interact, math, tool]","[second, grade, classroom, next, year, made, a...","[student, need, hand, practic, mathemat, fun, ...",interact math tool second grade classroom next...


In [28]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names[:30])

['00' '000' '00pm' '03' '04' '05' '10' '100' '1000' '100th' '101' '102'
 '103' '104' '105' '106' '107' '108' '10th' '11' '110' '1100' '112' '115'
 '11th' '12' '120' '1200' '123' '124']


# Model Building

In [29]:
# Train the model
X = vectorizer.fit_transform(df_donor["combined_text"])
y = df_donor["project_is_approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

# TF-IDF Embedding

In [30]:
# Create TF-IDF Vectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,3),   # unigram + bigram + trigram
    min_df=5,
    max_df=0.9
)

# Generate TF-IDF matrix

tfidf_matrix = tfidf_vectorizer.fit_transform(df_donor["combined_text"])

In [31]:
# creating TF-IDF vocabulary

tfidf_vocab = tfidf_vectorizer.vocabulary_

idf = dict(zip(
    tfidf_vectorizer.get_feature_names_out(),
    tfidf_vectorizer.idf_))

print("Vocabulary Size:", len(tfidf_vocab))

Vocabulary Size: 5000


In [32]:
def tfidf_weighted_document(tokens, model, idf):

    vectors = []
    weights = []

    for word in tokens:

        if word in model and word in idf:

            vectors.append(model[word])
            weights.append(idf[word])

    if len(vectors)==0:
        return np.zeros(model.vector_size)

    vectors = np.array(vectors)
    weights = np.array(weights)

    return np.average(vectors, axis=0,
                      weights=weights)

# Word2Vec Embedding

In [33]:
word2vec_model = api.load("word2vec-google-news-300")
print("Loaded Successfully")

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Loaded Successfully


In [34]:
# Creating the sentences variable
sentences = (
    df_donor["titles_tokens"].tolist() +
    df_donor["essays_tokens"].tolist() +
    df_donor["summary_tokens"].tolist())

print("Number of sentences:", len(sentences))
print("First sentence:", sentences[0])

Number of sentences: 327744
First sentence: ['educ', 'support', 'english', 'learner', 'home']


In [35]:
# Training the Word2Vec model

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=50,     # Embedding size
    window=3,            # Context window
    min_count=5,         # Ignore rare words
    workers=-1,           # CPU cores
    sg=1,                # 1 = Skip-Gram, 0 = CBOW
    epochs=3)

In [36]:
# Check the vocabulary

print("Vocabulary Size:", len(word2vec_model.wv))

Vocabulary Size: 15436


In [37]:
# Get the vector of a word

print(word2vec_model.wv["student"])

[-1.0724545e-03  4.7286271e-04  1.0206699e-02  1.8018546e-02
 -1.8605899e-02 -1.4233618e-02  1.2917745e-02  1.7945977e-02
 -1.0030856e-02 -7.5267432e-03  1.4761009e-02 -3.0669428e-03
 -9.0732267e-03  1.3108104e-02 -9.7203208e-03 -3.6320353e-03
  5.7531595e-03  1.9837476e-03 -1.6570430e-02 -1.8897636e-02
  1.4623532e-02  1.0140524e-02  1.3515387e-02  1.5257311e-03
  1.2701781e-02 -6.8107317e-03 -1.8928028e-03  1.1537147e-02
 -1.5043275e-02 -7.8722071e-03 -1.5023164e-02 -1.8600845e-03
  1.9076237e-02 -1.4638334e-02 -4.6675373e-03 -3.8754821e-03
  1.6154874e-02 -1.1861792e-02  9.0324880e-05 -9.5074680e-03
 -1.9207101e-02  1.0014586e-02 -1.7519170e-02 -8.7836506e-03
 -7.0199967e-05 -5.9236289e-04 -1.5322480e-02  1.9229487e-02
  9.9641159e-03  1.8466286e-02]


In [38]:
# Find similar words

word2vec_model.wv.most_similar("student", topn=10)

[('underr', 0.5570385456085205),
 ('moveabl', 0.5327945351600647),
 ('fastmath', 0.4907900393009186),
 ('booth', 0.4857129752635956),
 ('shred', 0.48262038826942444),
 ('topic', 0.4813556969165802),
 ('dian', 0.47297370433807373),
 ('push', 0.4684697389602661),
 ('merriam', 0.4672319293022156),
 ('walker', 0.4634425938129425)]

In [39]:
# Convert each document into an average Word2Vec embedding

def document_vector(tokens, model):

    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


# Create document embeddings

document_vectors = np.array([
    document_vector(tokens, word2vec_model)
    for tokens in df_donor["essays_tokens"]
])

print("Document Embeddings Shape:", document_vectors.shape)

Document Embeddings Shape: (109248, 50)


In [40]:
# Create Word2Vec features

df_donor["title_embedding"] = df_donor["titles_tokens"].apply(
    lambda x: document_vector(x, word2vec_model))

df_donor["essay_embedding"] = df_donor["essays_tokens"].apply(
    lambda x: document_vector(x, word2vec_model))

df_donor["summary_embedding"] = df_donor["summary_tokens"].apply(
    lambda x: document_vector(x, word2vec_model))

In [41]:
# combining into one feature vector

df_donor["w2v_vector"] = df_donor.apply(
    lambda row: np.concatenate([
        row["title_embedding"],
        row["essay_embedding"],
        row["summary_embedding"]
    ]),
    axis=1
)

In [42]:
# Convert vectors into a feature matrix

X_w2v = np.vstack(df_donor["w2v_vector"].values)

print(X_w2v.shape)

(109248, 150)


# Train-Test Split

In [43]:
# Split the original dataset

X = tfidf_matrix      # or X_w2v
y = df_donor["project_is_approved"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y)

In [44]:
# Verify the shapes

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (87398, 5000)
X_test : (21850, 5000)
y_train: (87398,)
y_test : (21850,)


# Classical Machine Learning Models

## Logistic Regression

In [45]:
# Train Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [46]:
# Prediction

y_pred_lr = lr_model.predict(X_test)

In [47]:
# Evaluation

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_lr))

print("\nClassification Report")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8505720823798627

Classification Report
              precision    recall  f1-score   support

           0       0.55      0.07      0.13      3308
           1       0.86      0.99      0.92     18542

    accuracy                           0.85     21850
   macro avg       0.70      0.53      0.52     21850
weighted avg       0.81      0.85      0.80     21850


Confusion Matrix
[[  244  3064]
 [  201 18341]]


## Random Forest

In [48]:
# Create the Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1)

In [49]:
# Train the model

rf_model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [50]:
# Prediction

y_pred_rf = rf_model.predict(X_test)

In [51]:
print("Random Forest Accuracy:",
      accuracy_score(y_test, y_pred_rf))

print("\nClassification Report")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.8475057208237986

Classification Report
              precision    recall  f1-score   support

           0       0.25      0.00      0.01      3308
           1       0.85      1.00      0.92     18542

    accuracy                           0.85     21850
   macro avg       0.55      0.50      0.46     21850
weighted avg       0.76      0.85      0.78     21850


Confusion Matrix
[[   12  3296]
 [   36 18506]]


## XGBoost

In [52]:
# Create the XGBoost model

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

In [53]:
# Train the model

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [54]:
# Prediction

y_pred_xgb = xgb_model.predict(X_test)

In [55]:
# Evaluation

print("XGBoost Accuracy:",
      accuracy_score(y_test, y_pred_xgb))

print("\nClassification Report")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Accuracy: 0.848649885583524

Classification Report
              precision    recall  f1-score   support

           0       0.51      0.01      0.02      3308
           1       0.85      1.00      0.92     18542

    accuracy                           0.85     21850
   macro avg       0.68      0.50      0.47     21850
weighted avg       0.80      0.85      0.78     21850


Confusion Matrix
[[   38  3270]
 [   37 18505]]


In [56]:
# Compare all models

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
         "Random Forest",
        "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)]})

comparison

,Model,Accuracy
0,Logistic Regression,0.850572
1,Random Forest,0.847506
2,XGBoost,0.848650


# Hyperparameter Tuning

In [59]:
# Logistic Regression Hyperparameter Tuning

lr = LogisticRegression(random_state=42)

lr_param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"],
    "max_iter": [500, 1000]}

grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=lr_param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1)

grid_lr.fit(X_train, y_train)

print("Best Parameters:", grid_lr.best_params_)
print("Best CV Accuracy:", grid_lr.best_score_)

# Best model
best_lr = grid_lr.best_estimator_

# Predictions
y_pred_best = best_lr.predict(X_test)

# Accuracy
print("Test Accuracy:", accuracy_score(y_test, y_pred_best))

# Classification Report
print("\nClassification Report")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_best))

Best Parameters: {'C': 1, 'max_iter': 500, 'solver': 'lbfgs'}
Best CV Accuracy: 0.8511407497431046
Test Accuracy: 0.8505720823798627

Classification Report
              precision    recall  f1-score   support

           0       0.55      0.07      0.13      3308
           1       0.86      0.99      0.92     18542

    accuracy                           0.85     21850
   macro avg       0.70      0.53      0.52     21850
weighted avg       0.81      0.85      0.80     21850


Confusion Matrix
[[  244  3064]
 [  201 18341]]


In [72]:
#  Random Forest Hyperparameter Tuning

# Create the model

rf = RandomForestClassifier(random_state=42)

# Parameter grid

rf_param = {'n_estimators':[100],
   'max_depth':[10, None],
   'min_samples_split':[2]}

grid_rf = GridSearchCV(
    estimator=rf, param_grid=rf_param,
  cv=2, scoring="accuracy", n_jobs=-1)

grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print("Best CV Accuracy:", grid_rf.best_score_)

# Best model

best_rf = grid_rf.best_estimator_

# Predictions

y_pred_best = best_rf.predict(X_test)

# Accuracy

print("Test Accuracy:", accuracy_score(y_test, y_pred_best))

# Classification Report

print("\nClassification Report")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_best))

Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Accuracy: 0.8485777706583675
Test Accuracy: 0.848604118993135

Classification Report
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      3308
           1       0.85      1.00      0.92     18542

    accuracy                           0.85     21850
   macro avg       0.42      0.50      0.46     21850
weighted avg       0.72      0.85      0.78     21850


Confusion Matrix
[[    0  3308]
 [    0 18542]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Deep Learning Model

In [77]:
# TF-IDF branch

tfidf_input = Input(shape=(5000,))

In [78]:
# TF-IDF Input Branch

# Input layer for TF-IDF features

tfidf_input = Input(shape=(5000,), name="TFIDF_Input")

# Hidden layer
branch1 = Dense(256,
    kernel_regularizer=l2(0.001)
)(tfidf_input)

branch1 = BatchNormalization()(branch1)
branch1 = LeakyReLU(alpha=0.1)(branch1)
branch1 = Dropout(0.4)(branch1)

# Second hidden layer
branch1 = Dense(128,
    kernel_regularizer=l2(0.001)
)(branch1)

branch1 = BatchNormalization()(branch1)
branch1 = LeakyReLU(alpha=0.1)(branch1)
branch1 = Dropout(0.3)(branch1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [79]:
# Word2Vec Input Branch

# Input layer for Word2Vec vectors
w2v_input = Input(shape=(300,), name="Word2Vec_Input")

# Hidden layer
branch2 = Dense(128,
    kernel_regularizer=l2(0.001)
)(w2v_input)

branch2 = BatchNormalization()(branch2)
branch2 = LeakyReLU(alpha=0.1)(branch2)
branch2 = Dropout(0.4)(branch2)

# Second hidden layer
branch2 = Dense(64,
    kernel_regularizer=l2(0.001)
)(branch2)

branch2 = BatchNormalization()(branch2)
branch2 = LeakyReLU(alpha=0.1)(branch2)
branch2 = Dropout(0.3)(branch2)

In [80]:
# Merge Both Branches

merged = Concatenate(name="Feature_Concatenation")(
    [branch1, branch2])

# Dense Layers After Merging

merged = Dense(128,
    kernel_regularizer=l2(0.001)
)(merged)

merged = BatchNormalization()(merged)
merged = LeakyReLU(alpha=0.1)(merged)
merged = Dropout(0.4)(merged)

merged = Dense(64,
    kernel_regularizer=l2(0.001)
)(merged)

merged = BatchNormalization()(merged)
merged = LeakyReLU(alpha=0.1)(merged)
merged = Dropout(0.3)(merged)

# Output Layer

output = Dense(1,
    activation="sigmoid",
    name="Output"
)(merged)

# Build Functional API Model

model = Model(
    inputs=[tfidf_input, w2v_input],
    outputs=output)

# Display Model Summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ TFIDF_Input         │ (None, 5000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Word2Vec_Input      │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │  1,280,256 │ TFIDF_Input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     38,528 │ Word2Vec_Input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 256)       │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 128)       │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ leaky_re_lu[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ leaky_re_lu_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 128)       │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_3       │ (None, 64)        │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ leaky_re_lu_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ leaky_re_lu_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Feature_Concatenat… │ (None, 192)       │          0 │ dropout_1[0][0],  │
│ (Concatenate)       │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,396,033 (5.33 MB)

 Trainable params: 1,394,497 (5.32 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [81]:
# Compile

model.compile(optimizer="adam",
loss="binary_crossentropy",
metrics=["accuracy"])

In [82]:
# Train

model.fit([X_train_tfidf.toarray(),X_train_w2v],
y_train, epochs=10, batch_size=32,
validation_split=0.2)

NameError: name 'X_train_tfidf' is not defined

# Performance Visualization

In [75]:
# Confusion Matrix

ConfusionMatrixDisplay

# ROC Curve

roc_curve()
auc()

NameError: name 'ConfusionMatrixDisplay' is not defined

In [ ]:
# Precision Recall Curve

precision_recall_curve()

# Loss Curve

plt.plot(history.history["loss"])

# Accuracy Curve

plt.plot(history.history["accuracy"])